# ML-04 — Search Intelligence Data Contract

This notebook frames the **Refresh / Content Opportunity Scoring** lane on the warehouse release. I am treating the slice as a content-month view built from March 2026 daily rows: one row is one client × content item observation for one month, and the output is a ranked refresh-review queue with a next-month decline proxy.

## 1. Unit of analysis + time window

One row = one pseudonymized client × content item for one month, rolled up from the daily warehouse fact. I use the March 2026 partition as the feature slice, and I reserve April 2026 only for the outcome window that defines the label.

In [7]:
%pip -q install duckdb huggingface_hub scikit-learn

import os
import getpass
from IPython.display import display

import duckdb
import pandas as pd

month = "2026-03"
next_month = "2026-04"

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH_FACT = f"read_parquet('{REL}/fact_content_daily_performance/month={month}/*.parquet')"
APRIL_FACT = f"read_parquet('{REL}/fact_content_daily_performance/month={next_month}/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

contract_summary = pd.DataFrame(
    [
        {
            "question": "What one row means",
            "answer": "One row is one client × content item monthly observation built from the March 2026 daily fact.",
        },
        {
            "question": "Which tables I use",
            "answer": "fact_content_daily_performance for the monthly slice, dim_content for metadata, and dim_clients only for coverage checks.",
        },
        {
            "question": "Which time window I use",
            "answer": "March 2026 is the feature window; April 2026 is the outcome window used only to define the label.",
        },
        {
            "question": "What I rank or predict",
            "answer": "I rank content items for refresh review by next-month decline risk.",
        },
        {
            "question": "What I deliberately exclude",
            "answer": "Future-window columns and IDs are context only; they do not go into the model features.",
        },
    ]
)

display(contract_summary)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


,question,answer
0,What one row means,One row is one client × content item monthly o...
1,Which tables I use,fact_content_daily_performance for the monthly...
2,Which time window I use,March 2026 is the feature window; April 2026 i...
3,What I rank or predict,I rank content items for refresh review by nex...
4,What I deliberately exclude,Future-window columns and IDs are context only...


## 2. Fields: feature / label / context / excluded

I keep the monthly signals that are knowable by the decision moment as features, the next-month decline flag as the label, the pseudonymized IDs and month key as context, and anything from April 2026 or later as excluded because it leaks the answer.

In [8]:
field_contract = pd.DataFrame(
    [
        {"field": "march_impressions", "bucket": "feature", "why": "Observed in the March 2026 slice before the review decision."},
        {"field": "march_ctr", "bucket": "feature", "why": "Computed from March clicks and impressions, so it is knowable at the decision moment."},
        {"field": "march_avg_position", "bucket": "feature", "why": "Search Console position is already observed for the March slice."},
        {"field": "content_age_days", "bucket": "feature", "why": "Content age is fixed metadata available before anyone acts."},
        {"field": "days_since_last_update", "bucket": "feature", "why": "Freshness is known from content metadata before refresh work starts."},
        {"field": "is_declining_next_month", "bucket": "label", "why": "This is the observed outcome I want to predict or rank against."},
        {"field": "client_hash_id", "bucket": "context", "why": "Needed for grouping and splitting, not for model learning."},
        {"field": "content_hash_id", "bucket": "context", "why": "Needed to identify the row and join tables, not as a predictor."},
        {"field": "report_month", "bucket": "context", "why": "Time key for the content-month slice, not a learning signal."},
        {"field": "ga4_data_available", "bucket": "excluded", "why": "This is a data-availability gate; I use it to filter rows, not to teach the model."},
        {"field": "april_impressions", "bucket": "excluded", "why": "Future-window information that would leak the target."},
    ]
)

display(field_contract)

,field,bucket,why
0,march_impressions,feature,Observed in the March 2026 slice before the re...
1,march_ctr,feature,"Computed from March clicks and impressions, so..."
2,march_avg_position,feature,Search Console position is already observed fo...
3,content_age_days,feature,Content age is fixed metadata available before...
4,days_since_last_update,feature,Freshness is known from content metadata befor...
5,is_declining_next_month,label,This is the observed outcome I want to predict...
6,client_hash_id,context,"Needed for grouping and splitting, not for mod..."
7,content_hash_id,context,"Needed to identify the row and join tables, no..."
8,report_month,context,"Time key for the content-month slice, not a le..."
9,ga4_data_available,excluded,This is a data-availability gate; I use it to ...


## 3. Verify it with queries (grain, counts, missing values, windows)

I use March 2026 as the proof slice. The first query checks the grain of the monthly rollup, the second checks how many raw daily rows land in the month and what dates they span, and the third filters with `IS TRUE` so the availability gate is explicit. After that I build the feature frame and run the leakage trap on the same slice.

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

grain_check = con.sql(f"""
WITH march_rollup AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks
    FROM {MARCH_FACT}
    WHERE ga4_data_available IS TRUE
    GROUP BY 1, 2
 )
SELECT
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_rows
FROM march_rollup
GROUP BY 1, 2
HAVING COUNT(*) > 1
LIMIT 5
""").df()
display(grain_check)

month_span = con.sql(f"""
SELECT
    COUNT(*) AS raw_rows,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM {MARCH_FACT}
""").df()
display(month_span)

availability_check = con.sql(f"""
SELECT
    COUNT(*) AS rows_surviving
FROM {MARCH_FACT}
WHERE ga4_data_available IS TRUE
""").df()
display(availability_check)

feature_frame = con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        DATE_TRUNC('month', report_date) AS report_month,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        CASE
            WHEN SUM(gsc_impressions) > 0 THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
            ELSE 0
        END AS march_ctr,
        AVG(gsc_avg_position) AS march_avg_position,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_days
    FROM {MARCH_FACT}
    GROUP BY 1, 2, 3
),
april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions
    FROM {APRIL_FACT}
    GROUP BY 1, 2
 )
SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.report_month,
    m.march_impressions,
    m.march_ctr,
    m.march_avg_position,
    c.content_age_days,
    c.days_since_last_update,
    a.april_impressions,
    CASE
        WHEN a.april_impressions < m.march_impressions * 0.8 THEN 1
        ELSE 0
    END AS is_declining_next_month
FROM march m
JOIN {DIM_CONTENT} c
    ON c.content_hash_id = m.content_hash_id
LEFT JOIN april a
    USING (client_hash_id, content_hash_id)
WHERE m.ga4_days > 0
""").df()

display(feature_frame.head(10))

feature_columns = [
    "march_impressions",
    "march_ctr",
    "march_avg_position",
    "content_age_days",
    "days_since_last_update",
]

model_frame = feature_frame.dropna(subset=feature_columns + ["is_declining_next_month"]).copy()

X_honest = model_frame[feature_columns]
y = model_frame["is_declining_next_month"].astype(int)

X_leaky = model_frame[feature_columns + ["is_declining_next_month"]].copy()

def score_frame(X: pd.DataFrame, y: pd.Series) -> dict[str, float]:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42,
        stratify=y,
    )
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    return {
        "accuracy": accuracy_score(y_test, predictions),
        "roc_auc": roc_auc_score(y_test, probabilities),
    }

honest_score = score_frame(X_honest, y)
leaky_score = score_frame(X_leaky, y)

score_comparison = pd.DataFrame(
    [
        {"setup": "honest features", **honest_score},
        {"setup": "label copy included", **leaky_score},
    ]
)

display(score_comparison)

del X_leaky

final_feature_frame = model_frame[feature_columns + ["is_declining_next_month"]].copy()
display(final_feature_frame.head(5))

HTTPException: HTTP Error: HTTP GET error on 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet' (HTTP 0 Internal Server Error)

## 4. Data limits

This slice cannot tell me whether a refresh actually causes recovery. It is also an unbalanced panel: different clients start at different times, and early rows can be GA4-zero-filled before `ga4_data_available` turns on. Any feature window that reaches into April 2026 or later would leak the answer, so I keep the decision slice and the outcome window separate.

In [ ]:
limits = pd.DataFrame(
    [
        {
            "limit": "Unbalanced history",
            "why_it_matters": "Some clients have much longer coverage than others, so March does not represent every client equally.",
        },
        {
            "limit": "GA4 zero-fill before availability",
            "why_it_matters": "Rows before `ga4_data_available` should be filtered, not treated as true zero traffic.",
        },
        {
            "limit": "Future-window leakage",
            "why_it_matters": "April outcomes must stay out of the March feature set or the score becomes unrealistically strong.",
        },
    ]
)

display(limits)

,limit,why_it_matters
0,Unbalanced history,Some clients have much longer coverage than ot...
1,GA4 zero-fill before availability,Rows before `ga4_data_available` should be fil...
2,Future-window leakage,April outcomes must stay out of the March feat...


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.